# 📖 Notebook 2: Idempotency & Exactly-Once Payments

The #1 rule of payment systems: **never charge a customer twice for the same purchase**.

But networks are unreliable — requests time out, connections drop, servers restart.  
When a merchant doesn't get a response, they *must* retry. Without idempotency, every retry risks a duplicate charge.

## Learning Objectives

By the end of this notebook you'll understand:
- What idempotency means and why it's essential for payments
- How an **idempotency key** prevents duplicate charges
- The database constraint that enforces exactly-once semantics
- How Redis can provide fast idempotency lookups

## 🛠️ Setup

Start the infrastructure first:

```bash
cd 06-system-designs/payment-system
docker compose up -d
```

### Kernel Selection

Select the `.venv` kernel in VS Code's kernel picker (top-right of notebook).  
If it doesn't appear, reload the window: `Cmd+Shift+P` → "Reload Window".

In [ ]:
import psycopg2
import psycopg2.extras
import redis
import json
import uuid
import time

DB_CONFIG = {
    "host": "localhost", "port": 5432,
    "database": "payment_demo", "user": "demo", "password": "demo"
}
REDIS_CONFIG = {"host": "localhost", "port": 6379, "decode_responses": True}

def get_db():
    return psycopg2.connect(**DB_CONFIG)

def get_redis():
    return redis.Redis(**REDIS_CONFIG)

conn = get_db(); r = get_redis()
print(f"✅ Postgres connected"); print(f"✅ Redis connected")
conn.close()

---
## 1. The Problem: Double Charges

Imagine this real-world scenario:

1. Customer clicks **"Pay $49.99"** on a merchant's website.
2. Merchant sends a charge request to our API.
3. We successfully charge the card... but the **response gets lost** (network timeout).
4. Merchant never receives the response → retries the same request.
5. 💥 Customer gets charged **$99.98** instead of **$49.99**!

This is not a hypothetical — it happens in production systems every day.  
Let's see it happen, then fix it.

In [ ]:
def create_payment_NO_idempotency(merchant_id, amount_cents, description):
    """BAD: Creates a new PaymentIntent every time — no duplicate protection!"""
    pi_id = f"pi_{uuid.uuid4().hex[:12]}"
    conn = get_db()
    cur = conn.cursor()
    cur.execute("""
        INSERT INTO payment_intents (id, merchant_id, amount_cents, currency, description, status)
        VALUES (%s, %s, %s, 'usd', %s, 'created')
        RETURNING id
    """, (pi_id, merchant_id, amount_cents, description))
    result = cur.fetchone()[0]
    conn.commit()
    cur.close(); conn.close()
    return result

# Simulate: merchant sends the same request 3 times (retries after timeouts)
print("=== WITHOUT idempotency (dangerous!) ===")
print("Merchant retries the same $49.99 charge 3 times...\n")

for attempt in range(1, 4):
    pi = create_payment_NO_idempotency("merch_001", 4999, "Order #X - Widget")
    print(f"  Attempt {attempt}: Created new PaymentIntent {pi} → customer will be charged again!")

# Check: how many payment intents did we create?
conn = get_db()
cur = conn.cursor()
cur.execute("SELECT COUNT(*) FROM payment_intents WHERE description = 'Order #X - Widget'")
count = cur.fetchone()[0]
print(f"\n💥 Result: {count} PaymentIntents created for the SAME order!")
print(f"   Customer would be charged ${4999 * count / 100:.2f} instead of $49.99")
cur.close(); conn.close()

---
## 2. The Solution: Idempotency Keys

An **idempotency key** is a unique string the merchant sends with every request.  
If we see the same `(merchant_id, idempotency_key)` pair again, we return the **original result** instead of creating a new charge.

The key insight: we enforce this with a **database UNIQUE constraint**:
```sql
UNIQUE (merchant_id, idempotency_key)
```

The database itself prevents duplicates — no race conditions, no distributed locks needed.

In [ ]:
def create_payment_with_idempotency(merchant_id, amount_cents, description, idempotency_key):
    """
    GOOD: Uses an idempotency key to prevent duplicate PaymentIntents.
    If the same (merchant_id, idempotency_key) is sent again, we return
    the existing PaymentIntent instead of creating a new one.
    """
    pi_id = f"pi_{uuid.uuid4().hex[:12]}"
    conn = get_db()
    cur = conn.cursor()

    try:
        cur.execute("""
            INSERT INTO payment_intents (id, merchant_id, amount_cents, currency, description, status, idempotency_key)
            VALUES (%s, %s, %s, 'usd', %s, 'created', %s)
            RETURNING id, status
        """, (pi_id, merchant_id, amount_cents, description, idempotency_key))
        result = cur.fetchone()
        conn.commit()
        return {"id": result[0], "status": result[1], "is_new": True}

    except psycopg2.errors.UniqueViolation:
        # The idempotency key was already used — return the existing record
        conn.rollback()
        cur.execute("""
            SELECT id, status FROM payment_intents
            WHERE merchant_id = %s AND idempotency_key = %s
        """, (merchant_id, idempotency_key))
        existing = cur.fetchone()
        return {"id": existing[0], "status": existing[1], "is_new": False}

    finally:
        cur.close(); conn.close()

# Simulate: merchant sends the SAME idempotency key 3 times
print("=== WITH idempotency (safe!) ===")
print("Merchant retries the same $49.99 charge 3 times with key 'order-X-safe'...\n")

for attempt in range(1, 4):
    result = create_payment_with_idempotency("merch_001", 4999, "Order #X-safe", "order-X-safe")
    label = "NEW" if result["is_new"] else "EXISTING (duplicate blocked)"
    print(f"  Attempt {attempt}: {label} → {result['id']}  status={result['status']}")

print("\n✅ Only ONE PaymentIntent was created, no matter how many retries!")

---
## 3. Prove It Under Concurrency (Sequential Retries Prove Nothing)

The loop above sends three retries **one after another**. Of course only one row
was created — the second call ran long after the first committed. That test
would also pass with a plain `SELECT … if not found: INSERT`, which is broken.

The interesting case is the one that actually happens: the merchant's HTTP
client times out and retries **while the original request is still in flight**,
and both land on different API servers at the same instant.

Let's fire 8 simultaneous requests with the same idempotency key and check the
only thing that matters: **exactly one PaymentIntent exists, and all 8 callers
were told about the same one.**

In [ ]:
from concurrent.futures import ThreadPoolExecutor
import threading

CONCURRENT_CALLERS = 8


def count_intents(merchant_id, key):
    conn = get_db()
    cur = conn.cursor()
    cur.execute(
        "SELECT COUNT(*) FROM payment_intents WHERE merchant_id = %s AND idempotency_key = %s",
        (merchant_id, key))
    n = cur.fetchone()[0]
    cur.close(); conn.close()
    return n


def race_idempotency(create_fn, rounds=4):
    """Fire CONCURRENT_CALLERS identical requests at the same instant."""
    for rnd in range(1, rounds + 1):
        key = f"race-{uuid.uuid4().hex[:10]}"
        gate = threading.Barrier(CONCURRENT_CALLERS)

        def caller(_):
            gate.wait()
            return create_fn("merch_001", 4999, "Concurrent retry demo", key)

        with ThreadPoolExecutor(max_workers=CONCURRENT_CALLERS) as pool:
            results = list(pool.map(caller, range(CONCURRENT_CALLERS)))

        ids = {res["id"] for res in results}
        created = sum(1 for res in results if res["is_new"])
        rows = count_intents("merch_001", key)
        print(f"  round {rnd}: rows_in_db={rows}  distinct_ids_returned={len(ids)}  "
              f"reported_new={created}")

        # The invariant. Anything else is a double charge.
        assert rows == 1, f"created {rows} PaymentIntents for one idempotency key!"
        assert len(ids) == 1, f"callers got different PaymentIntents: {ids}"
        assert created == 1, f"{created} callers think they created it"


print(f"🏁 {CONCURRENT_CALLERS} simultaneous requests, same idempotency key, 4 rounds")
print("=" * 74)
race_idempotency(create_payment_with_idempotency)
print()
print("✅ Every round: one row, one id, exactly one caller told 'is_new'.")
print()
print("   Why this works and SELECT-then-INSERT does not: all 8 INSERTs reach")
print("   the unique index. Postgres makes seven of them BLOCK until the winner")
print("   commits, then raises UniqueViolation — so by the time a loser runs its")
print("   SELECT, the winner's row is committed and visible. The database is")
print("   doing the mutual exclusion; our Python is only deciding what to return.")

---
## 4. The Trap: Same Key, Different Request

Here's the bug that survives every test above. The merchant reuses an
idempotency key — a bug on their side, a truncated hash, a recycled order id —
but sends a **different amount**:

```
POST /payment_intents  key=order-99  amount=$5.00    → creates pi_abc, $5.00
POST /payment_intents  key=order-99  amount=$500.00  → returns pi_abc,  ← !!
```

Our function happily returns the original `$5.00` intent and reports success.
The merchant now believes they charged `$500.00`. They ship the goods. Nobody
finds out until the month-end reconciliation.

**Stripe's behaviour is to refuse:** reusing a key with a different payload
returns a `400`, explicitly telling the caller they have a bug. The way to do
that is to store a **fingerprint of the request** alongside the key and compare
it on replay.

In [ ]:
import hashlib

# Demonstrate the bug with the function we just "proved" correct.
reuse_key = f"reuse-{uuid.uuid4().hex[:8]}"
first = create_payment_with_idempotency("merch_001", 500, "Small order", reuse_key)
second = create_payment_with_idempotency("merch_001", 50_000, "BIG order", reuse_key)

conn = get_db()
cur = conn.cursor()
cur.execute("SELECT amount_cents, description FROM payment_intents WHERE id = %s", (first["id"],))
amount, desc = cur.fetchone()
cur.close(); conn.close()

print("❌ Same key, different amount:")
print(f"   request 1: $5.00      → {first['id']}  is_new={first['is_new']}")
print(f"   request 2: $500.00    → {second['id']}  is_new={second['is_new']}")
print(f"   stored:    ${amount / 100:,.2f} ({desc})")
assert second["id"] == first["id"] and amount == 500
print()
print("   The caller asked to charge $500.00, got a success response, and we")
print("   stored $5.00. No error anywhere. This is worse than a double charge,")
print("   because a double charge is at least visible.")


# ── The fix: fingerprint the request and compare on replay ──────────────
def request_fingerprint(merchant_id, amount_cents, currency, description):
    """Hash the fields that define the request. Anything the caller can vary
    and that changes the money movement belongs in here."""
    payload = f"{merchant_id}|{amount_cents}|{currency}|{description}"
    return hashlib.sha256(payload.encode()).hexdigest()


class IdempotencyConflict(Exception):
    """Same key, different request — a client bug we must surface, not absorb."""


def create_payment_checked(merchant_id, amount_cents, description, idempotency_key):
    """Idempotent create that refuses a key replay with a different payload."""
    fingerprint = request_fingerprint(merchant_id, amount_cents, "usd", description)
    pi_id = f"pi_{uuid.uuid4().hex[:12]}"
    conn = get_db()
    cur = conn.cursor()
    try:
        try:
            cur.execute("""
                INSERT INTO payment_intents
                    (id, merchant_id, amount_cents, currency, description, status, idempotency_key)
                VALUES (%s, %s, %s, 'usd', %s, 'created', %s)
                RETURNING id, status
            """, (pi_id, merchant_id, amount_cents, description, idempotency_key))
            row = cur.fetchone()
            conn.commit()
            return {"id": row[0], "status": row[1], "is_new": True}
        except psycopg2.errors.UniqueViolation:
            conn.rollback()

        # Replay: rebuild the stored request's fingerprint and compare.
        cur.execute("""
            SELECT id, status, merchant_id, amount_cents, currency, description
            FROM payment_intents WHERE merchant_id = %s AND idempotency_key = %s
        """, (merchant_id, idempotency_key))
        existing = cur.fetchone()
        stored_fp = request_fingerprint(existing[2], existing[3], existing[4], existing[5])
        if stored_fp != fingerprint:
            raise IdempotencyConflict(
                f"idempotency key '{idempotency_key}' was already used with a "
                f"different request (stored ${existing[3] / 100:,.2f}, "
                f"you sent ${amount_cents / 100:,.2f})")
        return {"id": existing[0], "status": existing[1], "is_new": False}
    finally:
        cur.close(); conn.close()


print()
print("✅ With a request fingerprint:")
key2 = f"checked-{uuid.uuid4().hex[:8]}"
a = create_payment_checked("merch_001", 500, "Small order", key2)
print(f"   request 1: $5.00   → {a['id']}  is_new={a['is_new']}")
b = create_payment_checked("merch_001", 500, "Small order", key2)
print(f"   replay:    $5.00   → {b['id']}  is_new={b['is_new']}  (correct replay)")
assert b["id"] == a["id"] and not b["is_new"]
try:
    create_payment_checked("merch_001", 50_000, "BIG order", key2)
    raise AssertionError("expected an IdempotencyConflict")
except IdempotencyConflict as e:
    print(f"   request 2: $500.00 → 400 Bad Request: {e}")

# And it still holds under concurrency.
print()
print("Re-running the concurrency race against the checked version:")
race_idempotency(create_payment_checked, rounds=3)
print()
print("💡 Two honest caveats on the fingerprint:")
print("   • Choosing the fields is a judgement call. Too few and you miss real")
print("     conflicts; too many (a client-generated timestamp, say) and every")
print("     legitimate retry looks like a conflict.")
print("   • Storing the fingerprint in a column beats recomputing it from the")
print("     row: recomputing only works while the stored fields are exactly")
print("     what the request contained, which stops being true the moment")
print("     anything else can update the row.")

---
## 3. Speeding Up Idempotency Checks with Redis

The database UNIQUE constraint is the **safety net** — it always works, even under concurrent requests.  
But we can avoid hitting the database entirely for repeated requests by caching idempotency results in Redis.

The flow:
1. Merchant sends a request with `idempotency_key`.
2. Check Redis: if the key exists, return the cached result immediately.
3. If not in Redis, try inserting into Postgres.
4. On success, cache the result in Redis with a TTL (e.g., 24 hours).
5. On `UniqueViolation`, look up the existing record, cache it, and return it.

In [ ]:
IDEMPOTENCY_TTL = 86400  # 24 hours in seconds

def create_payment_fast_idempotency(merchant_id, amount_cents, description, idempotency_key):
    """
    Production-grade: Redis for speed, Postgres UNIQUE constraint for safety.
    """
    r = get_redis()
    cache_key = f"idempotency:{merchant_id}:{idempotency_key}"

    # Step 1: Check Redis cache first (fast path)
    cached = r.get(cache_key)
    if cached:
        data = json.loads(cached)
        print(f"  ⚡ Redis HIT — returning cached result for key '{idempotency_key}'")
        return data

    # Step 2: Not in cache — try Postgres insert
    pi_id = f"pi_{uuid.uuid4().hex[:12]}"
    conn = get_db()
    cur = conn.cursor()

    try:
        cur.execute("""
            INSERT INTO payment_intents (id, merchant_id, amount_cents, currency, description, status, idempotency_key)
            VALUES (%s, %s, %s, 'usd', %s, 'created', %s)
            RETURNING id, status
        """, (pi_id, merchant_id, amount_cents, description, idempotency_key))
        row = cur.fetchone()
        conn.commit()
        result = {"id": row[0], "status": row[1], "is_new": True}
        # Cache the new result
        r.setex(cache_key, IDEMPOTENCY_TTL, json.dumps(result))
        print(f"  🆕 Created new PaymentIntent (cached in Redis for 24h)")
        return result

    except psycopg2.errors.UniqueViolation:
        conn.rollback()
        cur.execute("""
            SELECT id, status FROM payment_intents
            WHERE merchant_id = %s AND idempotency_key = %s
        """, (merchant_id, idempotency_key))
        existing = cur.fetchone()
        result = {"id": existing[0], "status": existing[1], "is_new": False}
        # Cache for future lookups
        r.setex(cache_key, IDEMPOTENCY_TTL, json.dumps(result))
        print(f"  🛡️  Duplicate blocked by Postgres (now cached in Redis)")
        return result

    finally:
        cur.close(); conn.close()

# Demo: 5 retries with fast idempotency
print("=== Fast idempotency with Redis + Postgres ===")
print("Merchant sends 5 requests with the same key 'fast-demo-001'...\n")

for attempt in range(1, 6):
    print(f"  --- Attempt {attempt} ---")
    result = create_payment_fast_idempotency("merch_002", 7777, "Fast idempotency demo", "fast-demo-001")
    print(f"       → {result['id']}  status={result['status']}  new={result['is_new']}\n")

---
## 4. Measuring the Performance Difference

Let's see how much faster Redis-based idempotency is compared to always hitting Postgres.

In [ ]:
# Clear Redis cache so we start fresh
r = get_redis()
r.flushdb()

# Benchmark: Postgres-only idempotency
key_pg = f"bench-pg-{uuid.uuid4().hex[:8]}"
create_payment_with_idempotency("merch_001", 100, "bench", key_pg)  # first insert

start = time.time()
for _ in range(100):
    create_payment_with_idempotency("merch_001", 100, "bench", key_pg)
pg_time = time.time() - start

# Benchmark: Redis + Postgres idempotency
key_redis = f"bench-redis-{uuid.uuid4().hex[:8]}"
create_payment_fast_idempotency("merch_001", 100, "bench", key_redis)  # first insert + cache

start = time.time()
for _ in range(100):
    create_payment_fast_idempotency("merch_001", 100, "bench", key_redis)
redis_time = time.time() - start

print(f"\n📊 100 duplicate requests:")
print(f"   Postgres only : {pg_time*1000:.1f} ms  ({pg_time*10:.2f} ms per request)")
print(f"   Redis + PG    : {redis_time*1000:.1f} ms  ({redis_time*10:.2f} ms per request)")
print(f"   Speedup       : {pg_time/redis_time:.1f}x faster with Redis cache")

---
## 5. Edge Case: What If Redis and Postgres Disagree?

Redis is a **cache**, not the source of truth. What if Redis says "not seen" but Postgres says "already exists"?

This can happen if:
- Redis was restarted (cache cleared)
- The TTL expired but the idempotency key is still in Postgres

Our code handles this correctly because:
1. Redis miss → we try Postgres INSERT
2. Postgres raises `UniqueViolation` → we look up the existing record
3. We cache the result in Redis for next time

The Postgres UNIQUE constraint is the **ultimate safety net**. Redis just makes it faster.

In [ ]:
# Simulate Redis failure: flush the cache, then retry
r = get_redis()
test_key = f"edge-case-{uuid.uuid4().hex[:8]}"

# First request: creates in Postgres and caches in Redis
print("Step 1: Initial request")
result1 = create_payment_fast_idempotency("merch_003", 1500, "Edge case demo", test_key)
print(f"  → {result1}\n")

# Simulate Redis going down: flush all keys
r.flushdb()
print("Step 2: Redis flushed (simulating Redis restart)\n")

# Retry: Redis miss, but Postgres UNIQUE constraint saves us
print("Step 3: Retry after Redis flush")
result2 = create_payment_fast_idempotency("merch_003", 1500, "Edge case demo", test_key)
print(f"  → {result2}\n")

print(f"Same PaymentIntent returned? {result1['id'] == result2['id']}")
print("✅ Postgres is the safety net — even if Redis loses data, we never double-charge.")

---
## 6. Summary

| Concept | How It Works |
|---------|-------------|
| **Idempotency Key** | A unique string the merchant sends with each request |
| **UNIQUE Constraint** | `(merchant_id, idempotency_key)` in Postgres prevents duplicates at the DB level |
| **Redis Cache** | Speeds up duplicate detection — avoids hitting Postgres for repeated requests |
| **Safety Net** | Postgres is always the source of truth; Redis is just a performance optimization |

### Key Takeaways

1. **Idempotency is non-negotiable** in payment systems — without it, retries cause double charges.
2. **The database constraint is the safety net** — it works even under concurrent requests and race conditions.
3. **Redis accelerates lookups** but is never the only line of defense.
4. **Merchants provide the idempotency key** (usually their order ID) — this puts them in control of deduplication.

➡️  Next notebook: **Ledger & Double-Entry Bookkeeping**